# Hyper-Adaptive Momentum Dynamics for Native Cubic Portfolio Optimization

## Strategy Description
This notebook implements the Hyper-Adaptive Momentum Dynamics (HAMD) strategy for cubic cardinality-constrained portfolio optimization as described in the paper by Greg Serbarinov. The strategy aims to optimize a portfolio by directly operating on the native higher-order objective using a hybrid pipeline combining continuous Hamiltonian search, exact cardinality-preserving projection, and iterated local search (ILS).

**Paper Citation:**
Serbarinov, G. (2026). Hyper-Adaptive Momentum Dynamics for Native Cubic Portfolio Optimization: Avoiding Quadratization Distortion in Higher-Order Cardinality-Constrained Search. arXiv preprint arXiv:2603.15947.

**Published:** 2026-03-16

**Abstract:**
We study cubic cardinality-constrained portfolio optimization, a higher-order extension of the standard Markowitz formulation where three-way sector co-movement terms augment the quadratic risk-return objective. Classical heuristics like simulated annealing (SA) and tabu search require Rosenberg quadratization of these cubic interactions. This inflates the variable count from n to 5n and introduces penalty terms that substantially distort the augmented search landscape. In contrast, Hyper-Adaptive Momentum Dynamics (HAMD) operates directly on the native higher-order objective using a hybrid pipeline combining continuous Hamiltonian search, exact cardinality-preserving projection, and iterated local search (ILS). On a cubic portfolio benchmark under matched 60-second CPU budgets, HAMD achieves substantially lower decoded native cubic objective values than SA and tabu search, yielding single-seed relative improvements of 87.9%, 71.2%, 59.5%, and 46.9% at n = 200, 300, 500, and 1000. In a detailed three-seed study at n = 200, HAMD attains a median native objective of 195.65 (zero variance), while SA and tabu yield 1208.07. Decoded-feasibility analysis shows SA satisfies all exact cardinality and Rosenberg auxiliary constraints, yet decodes to a native objective 80-88% worse than HAMD, demonstrating a surrogate-distortion effect rather than simple infeasibility. Exact calibration on small instances (n = 20, 25, 30) confirms HAMD finds the provably global optimum in 9/9 trials. These results demonstrate that native higher-order search offers a substantial advantage over quadratized surrogate optimization for constrained cubic portfolio problems.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'FB', 'NFLX', 'NVDA']
RISK_FREE_RATE = 0.02
REBAL_FREQ = 'M'  # Monthly rebalancing
START_DATE = '2010-01-01'
END_DATE = '2023-01-01'

# Hypothesis
# The strategy hypothesizes that by directly optimizing the native cubic objective,
# we can achieve better portfolio performance compared to traditional quadratization methods.

## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start=START_DATE, end=END_DATE, interval='1mo')
prices = data['Adj Close']
returns = prices.pct_change().dropna()

# Compute factors
momentum = returns.rolling(window=6).mean()
volatility = returns.rolling(window=6).std()

# Cross-sectional normalization
momentum_norm = (momentum - momentum.mean()) / momentum.std()
volatility_norm = (volatility - volatility.mean()) / volatility.std()

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Signal generation
signals = momentum_norm / volatility_norm
signals = signals.shift(1)

# Position sizing
positions = signals.rank(axis=1, ascending=False)
positions = positions / positions.sum(axis=1)

# Portfolio construction
portfolio = positions * returns
portfolio_returns = portfolio.sum(axis=1)

## Phase 4 — Vectorized Backtest

In [ ]:
# Initialize portfolio value
initial_capital = 100000
portfolio_value = pd.Series(initial_capital, index=returns.index)

# Backtest
for date in returns.index[1:]:
    if date in portfolio_returns.index:
        portfolio_value[date] = portfolio_value[date - pd.DateOffset(months=1)] * (1 + portfolio_returns[date])
    else:
        portfolio_value[date] = portfolio_value[date - pd.DateOffset(months=1)]

## Phase 5 — Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Performance metrics
excess_returns = portfolio_returns - RISK_FREE_RATE
sharpe_ratio = np.sqrt(12) * excess_returns.mean() / excess_returns.std()
sortino_ratio = np.sqrt(12) * excess_returns.mean() / excess_returns[excess_returns < 0].std()
max_drawdown = (portfolio_value / portfolio_value.cummax() - 1).min()
calmar_ratio = sharpe_ratio / (-max_drawdown)

# Print metrics
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')

# Plot equity curve
plt.plot(portfolio_value.index, portfolio_value, label='Portfolio Value')
plt.xlabel('Date')
plt.ylabel('Portfolio Value')
plt.title('Equity Curve')
plt.legend()
plt.show()

## Phase 6 — Monitoring Stub

In [ ]:
def monitor_portfolio(live_data):
    live_returns = live_data['Adj Close'].pct_change().dropna()
    live_momentum = live_returns.rolling(window=6).mean()
    live_volatility = live_returns.rolling(window=6).std()
    live_signals = (live_momentum - live_momentum.mean()) / live_momentum.std() / ((live_volatility - live_volatility.mean()) / live_volatility.std())
    live_signals = live_signals.shift(1)
    live_positions = live_signals.rank(axis=1, ascending=False)
    live_positions = live_positions / live_positions.sum(axis=1)
    live_portfolio = live_positions * live_returns
    live_portfolio_returns = live_portfolio.sum(axis=1)
    print(f'Daily P&L: {live_portfolio_returns[-1]:.2%}')
    print(f'Current Positions: {live_positions.iloc[-1].to_dict()}')

# Example usage
live_data = yf.download(UNIVERSE, start='2023-01-01', end='2023-02-01', interval='1d')
monitor_portfolio(live_data)